In [2]:
import re
import unicodedata

In [16]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)
araconfig = os.getenv("araconfig")

In [4]:
from glob import glob

In [5]:
from pdf2image import convert_from_path, pdfinfo_from_path
import pytesseract

In [22]:
from google import genai 

In [ ]:
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils import embedding_functions


f:\MiniConda\envs\AiDev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
base_path="./sources"
pdf_files = glob(base_path+"/*.pdf")
print(pdf_files)

['./sources\\KYC cards_REV3_2020.pdf', './sources\\KYC Mobile payments_REV3_2020.pdf', './sources\\Mobile Money Regulation V3.pdf', './sources\\Prepaid Rules web v2.pdf']


In [13]:
def clean_text(text: str) -> str:
    if not text:
        return ""

    # 1. Normalize line endings and Unicode representations (fixes Arabic PDF ligatures)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = unicodedata.normalize('NFKC', text)

    # 2. Remove invisible BiDi marks, zero-width spaces, and non-breaking spaces
    text = re.sub(r'[\u200e\u200f\u202a-\u202e\ufeff\u200b-\u200d]', '', text)
    text = text.replace('\xa0', ' ')

    # 3. Strip Arabic Tatweel (Kashida) and Tashkeel (diacritics)
    text = re.sub(r'ـ+', '', text)
    text = re.sub(r'[\u064b-\u0652\u0670]', '', text)

    # 4. Remove recurring regulatory headers, titles, and site URLs
    boilerplate = (
        r'(?m)^\s*(?:وحدة\s+مكافحة\s+غسل\s+الأموال\s+وتمويل\s+الإرهاب|'
        r'وتمويل\s+الإرهاب\s+وحدة\s+مكافحة\s+غسل\s+الأموال|'
        r'جمهورية\s+مصر\s+العربية|'
        r'إجراءات\s+العناية\s+الواجبة\s+بعملاء(?:\s+خدمة\s+الدفع\s+باستخدام\s+الهاتف\s+المحمول|\s+خدمات\s+الدفع\s+باستخدام\s+البطاقات\s+المدفوعة\s+مقدما)?|'
        r'www\.mlcu\.org\.eg|'
        r'54\s+شارع\s+الجمهورية.*?|'
        r'info@cbe\.org\.eg.*?)\s*$'
    )
    text = re.sub(boilerplate, '', text)

    # 5. Remove legal notices and copyright disclaimers
    disclaimers = (
        r'جميع\s+الحقوق\s+محفوظة\s+للبنك\s+المركزي\s+المصري.*?(?:\n|$)|'
        r'All\s+Information,\s+Photos,\s+Charts\s+and\s+Designs.*?reserved\.?(?:\n|$)|'
        r'©\s*\d{4}\s*Central\s+Bank\s+of\s+Egypt.*?reserved\.?(?:\n|$)'
    )
    text = re.sub(disclaimers, '', text, flags=re.DOTALL | re.IGNORECASE)

    # 6. Remove Table of Contents dots and dotted page lines
    text = re.sub(r'\.{4,}', ' ', text)
    text = re.sub(r'(?m)^\s*[\d٠-٩]+\s*\.{2,}.*?$', '', text)

    # 7. Remove standalone page numbers (Eastern & Western Arabic numerals)
    text = re.sub(r'(?m)^\s*[\d٠-٩]+\s*$', '', text)

    # 8. Remove isolated bullet points, hyphens, and custom font symbols
    text = re.sub(r'(?m)^\s*[•●▪◦\*\-–—\uf02d]\s*$', '', text)

    # 9. Clean excessive horizontal spaces and blank lines
    text = re.sub(r'[^\S\n]+', ' ', text)
    text = re.sub(r'[^\S\n]+$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

In [14]:
def extract_arabic_text(pdf_path: str, pages=None, ocr_timeout=120) -> list:
    """Extract Arabic OCR text page by page and preserve page metadata.

    Args:
        pdf_path: PDF path.
        pages: None for all pages, one 1-based page number, or an inclusive
            1-based page range such as (2, 5).
        ocr_timeout: Maximum OCR time in seconds for each page.

    Returns:
        A list of dictionaries containing text and source metadata.
    """
    try:
        if not os.path.isfile(pdf_path):
            raise FileNotFoundError(f"PDF file not found: {pdf_path}")

        if not araconfig:
            raise ValueError("The 'araconfig' value was not loaded from .env")

        if not isinstance(ocr_timeout, (int, float)) or ocr_timeout <= 0:
            raise ValueError("ocr_timeout must be a positive number")

        if isinstance(pages, int):
            if pages < 1:
                raise ValueError("Page numbers must be 1 or greater")
            page_numbers = [pages]
        elif isinstance(pages, tuple):
            if len(pages) != 2:
                raise ValueError("Page range must be a tuple like (2, 5)")
            first_page, last_page = pages
            if (
                not isinstance(first_page, int)
                or not isinstance(last_page, int)
                or first_page < 1
                or last_page < first_page
            ):
                raise ValueError("Invalid page range")
            page_numbers = range(first_page, last_page + 1)
        elif pages is None:
            page_count = int(pdfinfo_from_path(pdf_path)["Pages"])
            page_numbers = range(1, page_count + 1)
        else:
            raise TypeError("pages must be None, an integer, or a tuple of two integers")

        documents = []

        for page_number in page_numbers:
            images = convert_from_path(
                pdf_path,
                dpi=300,
                first_page=page_number,
                last_page=page_number,
            )

            if not images:
                continue

            text = pytesseract.image_to_string(
                images[0],
                lang="ara",
                config=araconfig,
                timeout=ocr_timeout,
            )
            cleaned_text = clean_text(text)

            if cleaned_text:
                documents.append({
                    "text": cleaned_text,
                    "metadata": {
                        "source": os.path.basename(pdf_path),
                        "page": page_number,
                    },
                })

        return documents

    except FileNotFoundError as error:
        print(f"File error: {error}")
    except pytesseract.TesseractNotFoundError:
        print("Tesseract was not found in the AiDev environment")
    except RuntimeError as error:
        print(f"OCR timeout or Tesseract error: {error}")
    except Exception as error:
        print(f"OCR processing failed: {error}")

    return []

In [15]:
PAGES_TO_PROCESS = None  # Use (1, 3) for a quick test on the first three pages.

extracted_text = [
    extract_arabic_text(
        pdf_path,
        pages=PAGES_TO_PROCESS,
        ocr_timeout=120,
    )
    for pdf_path in pdf_files
]

documents = [
    page
    for pdf_documents in extracted_text
    for page in pdf_documents
]

print(documents[33])

{'text': './.5\n\n.1.5\n\n. 5\n\n. ٠.5\n\n. 5\n\nيراعى في حالة العملاء من الشركات أو المنشآت متناهية الصغر فهم هيكل الملكية\n\nوالسيطرة على العميلء؛ وتحديد المستفيدين الحقيقيين منه» على أن تشمل إجراءات\n\n"التعرف على هوية المستفيد ١ لحقيقي والتحقق منها" كل من:\n\n5 سم الأشخاص الطبيعيين الذين لهم حصة ملكية مسيطرة على الشركة أو المنشأة\n(ان وجد).\n\n5 االأشخاص الطبيعيين الذين ليس لهم حصة ملكية مسيطرة على الشركة أو\nالمنشأة ويمارسون سيطرة عليه من خلال أية وسائل أخرى (إن وجد).\n\nحالة عدم التوصل الى أشخاص ينطبق عليهم ما ورد بالبندين (أ) و(ب).\n\nيتعين على البنك التأكد من استيفاء بيانات نموذج طلب فتح حساب الخدمة بالكامل\n(نموذج موحد خاص بالخدمة صادر عن البنك)»؛ والتوقيع عليه من قبل العميل أو\nالشخص المصرح له بالتعامل (في حالة الشركات والمنشآت متناهية الصغر) وذلك أمام\n\nيتعين على البنك فهم الغرض من التعامل على الحساب وطبيعة ذلك التعامل» إلا أنه في\nالحالات التي يكون فيها الغرض من التعامل وطبيعته واضحا للبنك يجوز للبنك ادراجهما\nفى نموذج طلب فتح حساب الخدمة دون الحصول على معلومات أو مستندات 

In [17]:
from huggingface_hub import InferenceClient
import numpy as np

hf_client = InferenceClient(
    token=os.getenv("HF_TOKEN")
)


class RemoteHFEmbeddingFunction:
    def name(self):
        return "remote_huggingface"

    def _embed(self, texts):
        if isinstance(texts, str):
            texts = [texts]

        embeddings = hf_client.feature_extraction(
            texts,
            model="BAAI/bge-m3",
        )
        array = np.asarray(embeddings, dtype=np.float32)

        if array.ndim == 1:
            array = array.reshape(1, -1)

        return array.tolist()

    def __call__(self, input):
        return self._embed(input)

    def embed_query(self, input):
        return self._embed(input)


ef = RemoteHFEmbeddingFunction()

In [37]:
chromaclient = chromadb.PersistentClient(path="./chromadatabase")
# ef = embedding_functions.HuggingFaceEmbeddingFunction(api_key = os.getenv("HF_TOKEN"), model_name = "BAAI/bge-m3")
collectionnchroma = chromaclient.get_or_create_collection(name="rag_db", embedding_function=ef)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1200,
    chunk_overlap = 200,
    separators=["\n\n", "\n", "؟", "!", ".", " ", ""]
)

In [38]:
all_chunks = []
all_metadatas = []
all_ids = []

for doc in documents:
    raw_text = doc.get("text", "").strip()
    if not raw_text:
        continue

    splits = text_splitter.split_text(raw_text)
    meta = doc.get("metadata", {})
    doc_name = meta.get("source", "doc").replace(" ", "-")
    page_num = meta.get("page", 0)

    for idx, chunk in enumerate(splits, start=1):
        all_chunks.append(chunk)
        all_metadatas.append({"Source": doc_name, "page": page_num})
        all_ids.append(f"{doc_name}_p{page_num}_c{idx}")

print(f"Chunks: {len(all_chunks)}")
print(f"Unique IDs: {len(all_ids) == len(set(all_ids))}")

Chunks: 246
Unique IDs: True


In [39]:
batch_size=16
for i in range(0,len(all_chunks),batch_size):
    collectionnchroma.upsert(
        documents=all_chunks[i:i+batch_size],
        metadatas=all_metadatas[i:i+batch_size],
        ids=all_ids[i:i+batch_size]
    )

In [25]:
for doc in documents:
    if "Prepaid" in doc.get("source", "") and doc.get("page") == 19:
        print(doc["text"])

In [40]:
googles = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [41]:
def answer_gemini(query:str, collection, n_results:int = 1)->str:
    db = collection.query(query_texts=[query], n_results=n_results)
    docs = db["documents"][0]
    metas = db["metadatas"][0]

    context_entries = []
    for doc,meta in zip(docs,metas):
        src = meta.get("source") or meta.get("doc_name") or meta.get("Source") or meta.get("sources")  or "مستند غير محدد"
        pg = meta.get("page","?")
        context_entries.append(f"[المستند: {src} | صفحة: {pg}]\n{doc}")

    formatted_text = "\n\n---\n\n".join(context_entries)

    prompt = f"""أنت مستشار قانوني ومصرفي أول متخصص في اللوائح والتعليمات الرقابية الصادرة عن البنك المركزي المصري ووحدة مكافحة غسل الأموال وتمويل الإرهاب.

مهمتك: الإجابة عن استفسار المستخدم بأسلوب تقريري احترافي، دقيق، وشديد التنظيم، مع الالتزام التام بالقواعد التالية:

1. الالتزام الصارم بالسياق (Zero Hallucination):
- أجب فقط وحصرياً بالاعتماد على النصوص المتاحة في "السياق التنظيمي المرفق".
- لا تستنتج، لا تفترض، ولا تعتمد على أي معلومات مسبقة خارج السياق.
- إذا كانت هناك معلومة ناقصة أو انقطع النص قبل توضيحها (مثل انقطاع النص بعد عبارة "هو" دون ذكر المبلغ)، وضّح ذلك صراحة بعبارة: "هذه المعلومة غير مذكورة في المستندات المتاحة؛ لانقطاع النص في الوثيقة الأصلية".

2. قاعدة تدقيق الأرقام (أولوية التفقيط النصي على الأرقام الحسابية):
- نظراً لاحتمالية حدوث أخطاء انعكاس في التعرف الضوئي (OCR) على الأرقام، قارن دائماً بين الرموز الرقمية (Digits) والنص العربي المكتوب بالحروف (التفقيط، مثل: "عشرون ألف"، "مائة ألف").
- في حال وجود أي تعارض أو تشوه للأرقام (مثل ظهور "7٠٠٠٠١" مع "عشرون ألف"): اعتمد النص المكتوب بالحروف حصرًا وارفض الرموز الرقمية المشوهة تماماً.
- أعد كتابة الرقم بصيغته الحسابية القياسية الصحيحة التي تطابق التفقيط النصي، متبوعة بالتفقيط والعملة. 
  (مثال: إذا ورد "7٠٠٠٠١ (عشرون ألف)"، تُكتب في الإجابة: 20,000 (عشرون ألف) جنيه مصري).

3. الهيكلة وعرض البيانات:
- ابدأ مباشرة بتلخيص تنفيذي موجز للإجابة.
- نظّم الحدود والمبالغ والاشتراطات في نقاط نقطية محددة أو جدول منسق، مع ذكر العملة دوماً (جنيه مصري).
- أورد اسم المستند ورقم الصفحة بين قوسين بجانب كل حكم تنظيمي أو حد مالي استندت إليه (مثال: [المستند: Prepaid-Rules-web-v2.pdf | صفحة: 19]).

السياق التنظيمي المرفق:
{formatted_text}

استفسار المستخدم:
{query}
""" 

    response = googles.models.generate_content(
        model = "gemini-3.6-flash",
        contents= prompt
    )
    return response.text

In [ ]:
answer = answer_gemini("ما هي الحدود القصوى اليومية والشهرية ورصيد البطاقة المدفوعة مقدماً للأشخاص الطبيعيين؟", collectionnchroma)
print(answer)

**الملخص التنفيذي:**
بناءً على القواعد التنظيمية الصادرة بشأن البطاقات المدفوعة مقدماً، ونصوص التعليمات الرقابية المتاحة في المستند المرفق، نورد لكم الحدود القصوى المقررة للرصيد والتعاملات الشهرية للأشخاص الطبيعيين، مع الإشارة إلى وجود انقطاع نصي في المستند بشأن الحد الأقصى اليومي. وقد تم الاستناد كلياً إلى التفقيط النصي لتصحيح الأخطاء الرقمية الناتجة عن مسح (OCR) الوثيقة.

---

### تفاصيل الحدود القصوى للبطاقات المدفوعة مقدماً (للأشخاص الطبيعيين):

1. **الحد الأقصى لرصيد البطاقة:**
   * **الحد المالي:** 20,000 (عشرون ألف) جنيه مصري.
   * **ملاحظة تدقيقية:** (تم اعتماد التفقيط النصي "عشرون ألف" وتصحيح الرقم المشوه "7٠٠٠٠١").
   * **[المستند: Prepaid-Rules-web-v2.pdf | صفحة: 19]**

2. **الحد الأقصى اليومي للسحب والتحويل والمشتريات (للعميل الواحد بالبنك):**
   * **البيان:** هذه المعلومة غير مذكورة في المستندات المتاحة؛ لانقطاع النص في الوثيقة الأصلية بعد عبارة "هو".
   * **[المستند: Prepaid-Rules-web-v2.pdf | صفحة: 19]**

3. **الحد الأقصى الشهري للسحب والتحويل والمشتريات (للأشخاص الطبيعي